In [ ]:
'''
Why this approach?
Consistency (build_grid_from_aoi): By using the exact same grid generation code (or loading the specific GPKG file your climate script made), we guarantee that cell_id #5 in the climate file corresponds exactly to cell_id #5 in this land cover file.

Centroids vs. Intersection:

Intersection: Would calculate the % of Forest vs. Urban in every 11km cell. This is accurate but computationally heavy and complex to store (one row per cell per class).

Centroids (Selected Method): This matches your climate methodology. It simply asks: "What land cover is at the exact center of this grid cell?" This is standard for dominating a grid with categorical data and is much faster.

'''
# %%
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from pathlib import Path
from shapely.geometry import box
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# %%
# ------- 1. CONFIG & ENVIRONMENT -------
def load_environment():
    """Load environment variables and return them as a dictionary."""
    # Adjust path if necessary based on where you run this notebook
    load_dotenv(Path("../utils/.env")) 
    env_vars = {
        "EXTRACTEDDATASET_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER"),
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "GeoBoundaries": os.getenv("GeoBoundaries"),
        "EXTRACTEDGEOBOUNDARIES_FOLDER": os.getenv("EXTRACTEDGEOBOUNDARIES_FOLDER"),
        "EXTRACTED_CLIMATE_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER") + "/ClimateFeatures" # inferred
    }
    return env_vars

folders = load_environment()

# Input Paths
LANDCOVER_COMBINED = f"{folders['EXTRACTEDLANDCOVER_FOLDER']}/landcover_combined.geojson"
AOI_PATH = r"..\..\ExtractedDatasets\GeoBoundaries\AOI_DZA_TUN\aoi_dza_tun.gpkg" 

# Output Paths
OUT_DIR = folders['EXTRACTEDLANDCOVER_FOLDER']
GRID_PATH = os.path.join(folders['EXTRACTED_CLIMATE_FOLDER'], "grid_0p1.gpkg") # Tries to load the existing climate grid first

# Settings
GRID_RES_DEG = 0.1

# %%
# ------- 2. LOAD OR CREATE THE EXACT SAME GRID -------
# We strictly use the same logic as the climate script to ensure cell_ids match

def build_grid_from_aoi(aoi_path, res_deg=0.1):
    print(f"Building new grid from {aoi_path} with res={res_deg}...")
    aoi = gpd.read_file(aoi_path).to_crs("EPSG:4326")
    w, s, e, n = aoi.total_bounds
    
    # Create ranges
    xs = np.arange(w, e, res_deg)
    ys = np.arange(s, n, res_deg)
    
    # Create cells
    cells = [box(x, y, x + res_deg, y + res_deg) for x in xs for y in ys]
    grid_bbox = gpd.GeoDataFrame(geometry=cells, crs="EPSG:4326")
    
    # Intersect with AOI to keep only relevant cells
    grid = gpd.overlay(grid_bbox, aoi[["geometry"]], how="intersection", keep_geom_type=True)
    grid = grid.reset_index(drop=True).reset_index(names="cell_id")
    return grid

# Try to load existing grid from climate extraction to be 100% sure IDs match
if os.path.exists(GRID_PATH):
    print(f"Loading existing grid from: {GRID_PATH}")
    grid = gpd.read_file(GRID_PATH)
else:
    # Fallback: Rebuild it using the exact same logic
    print("Existing grid not found. Rebuilding to match climate logic...")
    grid = build_grid_from_aoi(AOI_PATH, res_deg=GRID_RES_DEG)

print(f"Grid loaded: {len(grid)} cells.")

# %%
# ------- 3. LOAD LAND COVER DATA -------
print("Loading combined land cover polygons...")
lc_gdf = gpd.read_file(LANDCOVER_COMBINED)

# Ensure CRS matches the grid (EPSG:4326)
if lc_gdf.crs != grid.crs:
    print(f"Reprojecting Land Cover from {lc_gdf.crs} to {grid.crs}...")
    lc_gdf = lc_gdf.to_crs(grid.crs)

# Optional: Clean columns to keep only what's necessary (adjust column names based on your specific shapefile attributes)
# Typically looking for 'gridcode', 'label', 'class', or 'id'
print("Land cover columns:", lc_gdf.columns)
# lc_gdf = lc_gdf[['geometry', 'gridcode', 'class_name']] # Uncomment and adjust if you want to filter columns

# %%
# ------- 4. SAMPLING STRATEGY (SPATIAL JOIN) -------
# The climate script sampled rasters at the centroid. 
# We will do the same: create centroids for the grid and find which polygon they fall into.

# A. Create Centroids
centroids = grid.copy()
centroids['geometry'] = centroids.geometry.centroid

# B. Spatial Join (Point in Polygon)
print("Performing spatial join (sampling land cover at grid centroids)...")
# 'inner' join keeps only centroids that fall INSIDE a land cover polygon
joined = gpd.sjoin(centroids, lc_gdf, how="left", predicate="within")

# %%
# ------- 5. CLEANUP AND SAVE -------

# Select relevant columns. 
# We need 'cell_id' from the grid, and the classification columns from the land cover.
# We drop 'geometry' to save as a tabular dataset (CSV/Parquet) like the climate data.

# Identify columns that came from the landcover dataset (excluding index_right which sjoin adds)
lc_cols = [c for c in joined.columns if c not in ['cell_id', 'geometry', 'index_right']]

final_df = pd.DataFrame(joined[['cell_id'] + lc_cols])

# Handle duplicates? 
# If a centroid falls on the border of two polygons, sjoin might create two rows.
# We drop duplicates to keep one dominant class per cell.
final_df = final_df.drop_duplicates(subset=['cell_id'])

print(f"Extracted {len(final_df)} rows.")
print(final_df.head())

# Save
out_parquet = f"{OUT_DIR}/landcover_0p1_grid.parquet"
out_csv = f"{OUT_DIR}/landcover_0p1_grid.csv"

final_df.to_parquet(out_parquet)
final_df.to_csv(out_csv, index=False)

print(f"✅ Saved grid-aligned land cover to:\n {out_parquet}")

# %%
# ------- 6. VISUALIZATION CHECK -------
# Let's plot the grid, colored by the land cover code (assuming a numeric column like 'gridcode' exists)
# If your column is named differently (e.g., 'dn', 'code'), change 'column_to_plot' below.

column_to_plot = lc_cols[0] # defaults to the first available attribute column

# Merge back to geometry for plotting
plot_gdf = grid.merge(final_df, on='cell_id', how='left')

fig, ax = plt.subplots(figsize=(10, 10))
plot_gdf.plot(column=column_to_plot, ax=ax, cmap='tab20', legend=True, 
              legend_kwds={'bbox_to_anchor': (1, 1)})
ax.set_title(f"Land Cover mapped to 0.1 Grid ({column_to_plot})")
plt.show()